# Item A

### Carregamento dos dados

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from medmnist import BloodMNIST

# Definir a transformação para converter as imagens em Tensores PyTorch [0.0, 1.0]
# Nota: O ToTensor() já divide automaticamente os pixels [0, 255] por 255.0
data_transform = transforms.Compose([
    transforms.ToTensor()
])

# Carregar os datasets oficiais do BloodMNIST (Treino, Validação e Teste)
# O parâmetro download=True baixa a base automaticamente caso não exista localmente
train_dataset = BloodMNIST(split='train', transform=data_transform, download=True)
val_dataset   = BloodMNIST(split='val', transform=data_transform, download=True)
test_dataset  = BloodMNIST(split='test', transform=data_transform, download=True)

# Criar os DataLoaders para gerenciar o mini-batching
#BATCH_SIZE = 64  # Ou 128
BATCH_SIZE = 64

train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# gpu?
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### Treino do Modelo

In [8]:
import ml3.mlp_a as mlp_a
import pandas as pd

#hidden_dims=[128, 256, 512]
#lrs=[1e-3, 1e-4]
#dropouts=[0.0, 0.2, 0.5]

hidden_dims=[128, 256, 512]
lrs=[1e-3]
dropouts=[0.0, 0.2, 0.3, 0.5]

results = mlp_a.fit_mlp(
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    hidden_dims=hidden_dims,
    lrs=lrs,
    dropouts=dropouts,
    save_tag="_mlp_item_a"
)

df_results = pd.DataFrame(results)


--- Testando: hidden_dim=128, lr=0.001, dropout=0.0 ---

--- Testando: hidden_dim=128, lr=0.001, dropout=0.2 ---

--- Testando: hidden_dim=128, lr=0.001, dropout=0.3 ---

--- Testando: hidden_dim=128, lr=0.001, dropout=0.5 ---

--- Testando: hidden_dim=256, lr=0.001, dropout=0.0 ---

--- Testando: hidden_dim=256, lr=0.001, dropout=0.2 ---

--- Testando: hidden_dim=256, lr=0.001, dropout=0.3 ---

--- Testando: hidden_dim=256, lr=0.001, dropout=0.5 ---

--- Testando: hidden_dim=512, lr=0.001, dropout=0.0 ---

--- Testando: hidden_dim=512, lr=0.001, dropout=0.2 ---

--- Testando: hidden_dim=512, lr=0.001, dropout=0.3 ---

--- Testando: hidden_dim=512, lr=0.001, dropout=0.5 ---

Melhor Configuração Encontrada: {'hidden_dim': 512, 'lr': 0.001, 'dropout': 0.0}
Acurácia Balanceada na Validação: 0.7979


### Plotagem da Treino (Validação)

In [9]:
import plotly.express as px
import importlib
importlib.reload(pd)

# Ordenar pelos melhores resultados de Acurácia de Validação
df_sorted = df_results.sort_values(by="val_acc", ascending=False).reset_index(drop=True)

# Aplicar o estilo com gradiente de cor
styled_df = df_sorted.style.background_gradient(
    cmap="Blues", subset=["val_acc"]
).format({
    "val_acc": "{:.4f}",
    "lr": "{:.4f}",
    "dropout": "{:.1f}"
}).set_caption("<b>Desempenho da MLP na Busca por Hiperparâmetros</b>")

# Exibir a tabela no Notebook
styled_df

,hidden_dim,lr,dropout,val_acc
0,512,0.0010,0.0,0.7979
1,512,0.0010,0.2,0.7910
2,512,0.0010,0.3,0.7870
3,256,0.0010,0.0,0.7796
4,256,0.0010,0.2,0.7752
5,256,0.0010,0.3,0.7726
6,128,0.0010,0.0,0.7664
7,128,0.0010,0.3,0.7628
8,128,0.0010,0.2,0.7598
9,256,0.0010,0.5,0.7496


In [11]:
# Criar o gráfico de coordenadas paralelas
fig = px.parallel_coordinates(
    df_results,
    dimensions=['hidden_dim', 'lr', 'dropout', 'val_acc'],
    color='val_acc',
    #color_continuous_scale=px.colors.diverging.Tealrose,
    labels={
        'hidden_dim': 'Neurônios Ocultos',
        'lr': 'Taxa de Aprendizado (LR)',
        'dropout': 'Dropout (p)',
        'val_acc': 'Acurácia Bal. (Val)'
    },
    title="Exploração de Hiperparâmetros da MLP - Coordenadas Paralelas"
)

# Ajustar layout para exibição no Notebook
fig.update_layout(template="plotly_white")
fig.show()

# Linhas mais claras/vivas: Representam os experimentos com maior acurácia de validação.
# Trajetória: Permite identificar visualmente se uma determinada escolha (por exemplo,
#   lr = 0.001 vs 0.0001 ou dropout = 0.2) concentra as linhas que atingem as maiores
#   pontuações no eixo da direita (val_acc).

### Teste

In [17]:
import torch
import plotly.express as px
import ml3.mlp_a as mlp_a

# Instanciar o melhor modelo e carregar os pesos salvos
best_model = mlp_a.BloodMNISTMLP(hidden_dim=512, dropout_prob=0.0)
best_model.load_state_dict(torch.load('best_model_mlp_item_a.pth'))

res = mlp_a.evaluate_model(best_model, test_loader, device=DEVICE)

# Exibir Métricas
print(f"Acurácia Balanceada (Teste): {res['bal_acc']:.4f}")
print(f"F1-Score Macro (Teste)     : {res['f1_macro']:.4f}")

# 4. Plotar Matriz de Confusão
class_names = [
    'Basófilo', 'Eosinófilo', 'Eritroblasto', 'Granulócito Imaturo',
    'Linfócito', 'Monócito', 'Neutrófilo', 'Plaqueta'
]

fig_cm = px.imshow(
    res['cm'],
    x=class_names,
    y=class_names,
    text_auto=True,
    color_continuous_scale='Blues',
    title="<b>Matriz de Confusão - MLP (Teste BloodMNIST)</b>",
    labels=dict(x="Classe Predita", y="Classe Real", color="Frequência")
)
fig_cm.update_layout(template="plotly_white", width=700, height=600)
fig_cm.show()

Acurácia Balanceada (Teste): 0.7535
F1-Score Macro (Teste)     : 0.7618
